# Advanced Python Dictionary Views
## 15 Problems with Complete Solutions, Tests, and Best Practices

This notebook expands the uploaded lesson on `dict.keys()`, `dict.values()`, and
`dict.items()` into an advanced practice set.

You will work with:

- live/dynamic views;
- repeatable iterables versus one-shot iterators;
- set-like key and item operations;
- hashability constraints;
- deterministic output;
- safe mutation;
- missing keys versus falsy values;
- schema validation;
- merge policies;
- audits, diffs, and reconciliation.

Every problem includes a runnable solution and assertions.

## Best-practice rules used throughout

1. Iterate directly over a dictionary when only keys are needed.
2. Use `.items()` when both key and value are needed.
3. Remember that views are live, not snapshots.
4. Materialize a view only when a snapshot is intentional.
5. Never depend on the order of a set-operation result.
6. Do not change dictionary size while directly iterating over it.
7. Do not use `d1.get(key) or d2.get(key)` to choose a value.
8. Use membership tests or a unique sentinel when `None` is a valid value.
9. `values()` is not set-like.
10. `items()` set operations require hashable values.

## Extended examples before the problems

In [1]:
d = {"a": 1, "b": 2}

keys_view = d.keys()
values_view = d.values()
items_view = d.items()

print("Before:", list(keys_view), list(values_view), list(items_view))

d["c"] = 3
d["a"] = 10
del d["b"]

print("After: ", list(keys_view), list(values_view), list(items_view))

assert list(keys_view) == ["a", "c"]
assert list(values_view) == [10, 3]
assert list(items_view) == [("a", 10), ("c", 3)]

Before: ['a', 'b'] [1, 2] [('a', 1), ('b', 2)]
After:  ['a', 'c'] [10, 3] [('a', 10), ('c', 3)]


In [2]:
left = {"a": 1, "b": 2, "c": 3}
right = {"b": 20, "c": 30, "d": 40}

print("union:", sorted(left.keys() | right.keys()))
print("intersection:", sorted(left.keys() & right.keys()))
print("left only:", sorted(left.keys() - right.keys()))
print("right only:", sorted(right.keys() - left.keys()))
print("exclusive:", sorted(left.keys() ^ right.keys()))

union: ['a', 'b', 'c', 'd']
intersection: ['b', 'c']
left only: ['a']
right only: ['d']
exclusive: ['a', 'd']


In [3]:
view = left.keys()
iterator = iter(view)

print("view pass 1:", list(view))
print("view pass 2:", list(view))
print("iterator pass 1:", list(iterator))
print("iterator pass 2:", list(iterator))

assert list(view) == ["a", "b", "c"]
assert list(iterator) == []

view pass 1: ['a', 'b', 'c']
view pass 2: ['a', 'b', 'c']
iterator pass 1: ['a', 'b', 'c']
iterator pass 2: []


# Problem 1 — Live views versus snapshots


Retain a key view and a list snapshot. Mutate the dictionary by adding and
deleting keys. Demonstrate that the view changes while the snapshot does not.

## Solution 1

In [4]:
record = {"id": 101, "status": "new", "priority": 2}

live_keys = record.keys()
snapshot_keys = list(record.keys())

record["owner"] = "Mira"
del record["priority"]

print("live:", list(live_keys))
print("snapshot:", snapshot_keys)

assert list(live_keys) == ["id", "status", "owner"]
assert snapshot_keys == ["id", "status", "priority"]

live: ['id', 'status', 'owner']
snapshot: ['id', 'status', 'priority']


A dictionary view retains a reference to the dictionary. A list is an independent
snapshot created at one moment in time.

# Problem 2 — Ordered common-key report


Return `(key, left_value, right_value)` rows for shared keys. Preserve the
insertion order of the left dictionary, even though intersection returns a set.

## Solution 2

In [5]:
def common_key_report(left, right):
    common = left.keys() & right.keys()
    return [
        (key, left[key], right[key])
        for key in left
        if key in common
    ]


old_prices = {
    "coffee": 3.50,
    "tea": 2.80,
    "juice": 4.20,
    "water": 1.25,
}
new_prices = {
    "water": 1.30,
    "coffee": 3.75,
    "soda": 2.20,
    "tea": 2.80,
}

report = common_key_report(old_prices, new_prices)
print(report)

assert report == [
    ("coffee", 3.50, 3.75),
    ("tea", 2.80, 2.80),
    ("water", 1.25, 1.30),
]

[('coffee', 3.5, 3.75), ('tea', 2.8, 2.8), ('water', 1.25, 1.3)]


Use the set intersection for efficient membership discovery, then iterate the
source dictionary whose order should control the report.

# Problem 3 — Exclusive keys without the falsy-value bug


Build a dictionary containing keys found in exactly one input. Preserve
left-only order followed by right-only order. Values may be `0`, `False`, `""`,
or `None`.

## Solution 3

In [6]:
def exclusive_items(left, right):
    result = {}

    for key in left:
        if key not in right:
            result[key] = left[key]

    for key in right:
        if key not in left:
            result[key] = right[key]

    return result


left_data = {
    "zero": 0,
    "false": False,
    "shared": "left",
}
right_data = {
    "shared": "right",
    "empty": "",
    "none": None,
}

exclusive = exclusive_items(left_data, right_data)
print(exclusive)

assert exclusive == {
    "zero": 0,
    "false": False,
    "empty": "",
    "none": None,
}

{'zero': 0, 'false': False, 'empty': '', 'none': None}


Avoid `left.get(key) or right.get(key)`. Truthiness is not the same as key
presence, and all the falsy values in this test are legitimate stored values.

# Problem 4 — Classify key relationships


Classify two dictionaries into `only_left`, `only_right`, `same_value`, and
`changed_value`. All lists must be deterministic.

## Solution 4

In [7]:
def classify_changes(left, right):
    common = left.keys() & right.keys()

    return {
        "only_left": [key for key in left if key not in right],
        "only_right": [key for key in right if key not in left],
        "same_value": [
            key for key in left
            if key in common and left[key] == right[key]
        ],
        "changed_value": [
            key for key in left
            if key in common and left[key] != right[key]
        ],
    }


before = {
    "host": "db-01",
    "port": 5432,
    "ssl": True,
    "timeout": 30,
}
after = {
    "host": "db-02",
    "port": 5432,
    "ssl": True,
    "pool_size": 20,
}

classification = classify_changes(before, after)
print(classification)

assert classification == {
    "only_left": ["timeout"],
    "only_right": ["pool_size"],
    "same_value": ["port", "ssl"],
    "changed_value": ["host"],
}

{'only_left': ['timeout'], 'only_right': ['pool_size'], 'same_value': ['port', 'ssl'], 'changed_value': ['host']}


Key-view algebra identifies relationships; source iteration preserves stable,
meaningful order.

# Problem 5 — Structured mapping diff


Return added items, removed items, changed items with before/after values, and
unchanged keys.

## Solution 5

In [8]:
def diff_mappings(before, after):
    common = before.keys() & after.keys()

    return {
        "added": {
            key: after[key]
            for key in after
            if key not in before
        },
        "removed": {
            key: before[key]
            for key in before
            if key not in after
        },
        "changed": {
            key: {"before": before[key], "after": after[key]}
            for key in before
            if key in common and before[key] != after[key]
        },
        "unchanged": [
            key for key in before
            if key in common and before[key] == after[key]
        ],
    }


old_settings = {
    "theme": "light",
    "page_size": 25,
    "email_alerts": True,
    "timezone": "UTC",
}
new_settings = {
    "theme": "dark",
    "page_size": 25,
    "email_alerts": False,
    "language": "en",
}

mapping_diff = diff_mappings(old_settings, new_settings)
print(mapping_diff)

assert mapping_diff == {
    "added": {"language": "en"},
    "removed": {"timezone": "UTC"},
    "changed": {
        "theme": {"before": "light", "after": "dark"},
        "email_alerts": {"before": True, "after": False},
    },
    "unchanged": ["page_size"],
}

{'added': {'language': 'en'}, 'removed': {'timezone': 'UTC'}, 'changed': {'theme': {'before': 'light', 'after': 'dark'}, 'email_alerts': {'before': True, 'after': False}}, 'unchanged': ['page_size']}


This shape is useful for audit logs, configuration reviews, API change reports,
and patch generation.

# Problem 6 — Conflict-aware merge


Merge under three policies: left wins, right wins, or raise on conflicting
shared values. Identical shared values are not conflicts.

## Solution 6

In [9]:
def merge_with_policy(left, right, policy="error"):
    if policy not in {"left", "right", "error"}:
        raise ValueError("policy must be 'left', 'right', or 'error'")

    result = dict(left)
    common = left.keys() & right.keys()

    conflicts = [
        key for key in left
        if key in common and left[key] != right[key]
    ]

    if policy == "error" and conflicts:
        details = {
            key: (left[key], right[key])
            for key in conflicts
        }
        raise ValueError(f"Conflicting values: {details}")

    if policy == "right":
        for key in conflicts:
            result[key] = right[key]

    for key in right:
        if key not in left:
            result[key] = right[key]

    return result


base = {"region": "eu", "replicas": 2, "debug": False}
override = {"replicas": 4, "debug": False, "cache": True}

left_wins = merge_with_policy(base, override, "left")
right_wins = merge_with_policy(base, override, "right")

print(left_wins)
print(right_wins)

assert left_wins == {
    "region": "eu",
    "replicas": 2,
    "debug": False,
    "cache": True,
}
assert right_wins == {
    "region": "eu",
    "replicas": 4,
    "debug": False,
    "cache": True,
}

try:
    merge_with_policy(base, override, "error")
except ValueError as exc:
    print("expected:", exc)

{'region': 'eu', 'replicas': 2, 'debug': False, 'cache': True}
{'region': 'eu', 'replicas': 4, 'debug': False, 'cache': True}
expected: Conflicting values: {'replicas': (2, 4)}


An explicit merge policy is clearer and safer than silently overwriting values.

# Problem 7 — Item-view algebra and hashability


Find exact shared items and first-only items when values are hashable. Then
demonstrate why list values prevent item-view set operations.

## Solution 7

In [10]:
first = {"a": 1, "b": 2, "c": 3}
second = {"b": 2, "c": 30, "d": 4}

shared_items = first.items() & second.items()
first_only_items = first.items() - second.items()

print("shared:", sorted(shared_items))
print("first only:", sorted(first_only_items))

assert shared_items == {("b", 2)}
assert first_only_items == {("a", 1), ("c", 3)}

lists_1 = {"a": [1, 2], "b": [3, 4]}
lists_2 = {"b": [3, 4], "c": [5, 6]}

try:
    lists_1.items() & lists_2.items()
except TypeError as exc:
    print("expected:", type(exc).__name__, exc)

shared: [('b', 2)]
first only: [('a', 1), ('c', 3)]
expected: TypeError unhashable type: 'list'


An item is a tuple. A tuple is hashable only when all its elements are hashable.
Dictionary keys are hashable, but dictionary values are not guaranteed to be.

# Problem 8 — Compare values as multisets


`dict_values` is not set-like, and duplicate counts matter. Compare the value
collections of two mappings as multisets.

## Solution 8

In [11]:
from collections import Counter


def same_value_multiset(left, right):
    return Counter(left.values()) == Counter(right.values())


v1 = {"a": 1, "b": 1, "c": 2}
v2 = {"x": 2, "y": 1, "z": 1}
v3 = {"x": 2, "y": 2, "z": 1}

assert same_value_multiset(v1, v2) is True
assert same_value_multiset(v1, v3) is False

print("v1 and v2:", same_value_multiset(v1, v2))
print("v1 and v3:", same_value_multiset(v1, v3))

v1 and v2: True
v1 and v3: False


`Counter` preserves multiplicity. Converting values to a set would incorrectly
discard duplicates.

# Problem 9 — Support nested unhashable values


Extend multiset comparison to common nested containers by recursively converting
them to canonical hashable forms.

## Solution 9

In [12]:
from collections import Counter


def freeze(value):
    if isinstance(value, dict):
        return (
            "__dict__",
            tuple(sorted((freeze(k), freeze(v)) for k, v in value.items())),
        )
    if isinstance(value, list):
        return ("__list__", tuple(freeze(item) for item in value))
    if isinstance(value, tuple):
        return ("__tuple__", tuple(freeze(item) for item in value))
    if isinstance(value, set):
        return ("__set__", tuple(sorted(freeze(item) for item in value)))
    return value


def same_value_multiset_general(left, right):
    left_counter = Counter(freeze(value) for value in left.values())
    right_counter = Counter(freeze(value) for value in right.values())
    return left_counter == right_counter


complex_1 = {
    "a": [1, 2],
    "b": {"x": 10, "y": [20, 30]},
}
complex_2 = {
    "other_b": {"y": [20, 30], "x": 10},
    "other_a": [1, 2],
}

assert same_value_multiset_general(complex_1, complex_2)
print("Nested unhashable values compare correctly.")

Nested unhashable values compare correctly.


Canonicalization is useful for controlled data structures. For arbitrary custom
objects, define domain-specific normalization rather than guessing.

# Problem 10 — Safe mutation during iteration


Remove every negative-valued item using two safe strategies. Also demonstrate
the unsafe direct-deletion pattern.

## Solution 10

In [13]:
unsafe = {"a": 5, "b": -1, "c": 3, "d": -7}

try:
    for key in unsafe:
        if unsafe[key] < 0:
            del unsafe[key]
except RuntimeError as exc:
    print("expected:", type(exc).__name__, exc)


def remove_negatives_snapshot(data):
    for key, value in list(data.items()):
        if value < 0:
            del data[key]


def remove_negatives_two_phase(data):
    keys_to_delete = [
        key for key, value in data.items()
        if value < 0
    ]
    for key in keys_to_delete:
        del data[key]


data_1 = {"a": 5, "b": -1, "c": 3, "d": -7}
data_2 = {"a": 5, "b": -1, "c": 3, "d": -7}

remove_negatives_snapshot(data_1)
remove_negatives_two_phase(data_2)

assert data_1 == {"a": 5, "c": 3}
assert data_2 == {"a": 5, "c": 3}

print(data_1)
print(data_2)

expected: RuntimeError dictionary changed size during iteration
{'a': 5, 'c': 3}
{'a': 5, 'c': 3}


Changing dictionary size while directly iterating invalidates the traversal.
Snapshot iteration and two-phase deletion are both clear and safe.

# Problem 11 — Schema validation with key relations


Validate required and optional keys. Return missing and unexpected keys in
sorted order, plus a boolean result.

## Solution 11

In [14]:
def validate_schema(record, required, optional=()):
    required = set(required)
    allowed = required | set(optional)
    keys = record.keys()

    missing = required - keys
    unexpected = keys - allowed

    return {
        "valid": not missing and not unexpected,
        "missing": sorted(missing),
        "unexpected": sorted(unexpected),
    }


user_record = {
    "id": 7,
    "name": "Ada",
    "email": "ada@example.test",
    "debug_flag": True,
}

schema_result = validate_schema(
    user_record,
    required={"id", "name", "email", "role"},
    optional={"phone", "timezone"},
)

print(schema_result)

assert schema_result == {
    "valid": False,
    "missing": ["role"],
    "unexpected": ["debug_flag"],
}

required = {"id", "name"}
record = {"id": 1, "name": "Lin", "email": "lin@example.test"}

assert required <= record.keys()
assert record.keys() > required
assert record.keys().isdisjoint({"password", "token"})

{'valid': False, 'missing': ['role'], 'unexpected': ['debug_flag']}


Key views support subset, superset, and disjointness checks, making them useful
for validation and security rules.

# Problem 12 — Inventory reconciliation


Combine inventory dictionaries while preserving source order. Report quantities
from each warehouse, total quantity, and key-presence status.

## Solution 12

In [15]:
def reconcile_inventory(warehouse_a, warehouse_b):
    ordered_skus = list(warehouse_a)
    ordered_skus.extend(
        sku for sku in warehouse_b
        if sku not in warehouse_a
    )

    common = warehouse_a.keys() & warehouse_b.keys()
    result = {}

    for sku in ordered_skus:
        quantity_a = warehouse_a.get(sku, 0)
        quantity_b = warehouse_b.get(sku, 0)

        if sku in common:
            status = "both"
        elif sku in warehouse_a:
            status = "only_a"
        else:
            status = "only_b"

        result[sku] = {
            "warehouse_a": quantity_a,
            "warehouse_b": quantity_b,
            "total": quantity_a + quantity_b,
            "status": status,
        }

    return result


warehouse_a = {
    "SKU-100": 12,
    "SKU-200": 0,
    "SKU-300": 7,
}
warehouse_b = {
    "SKU-300": 5,
    "SKU-100": 8,
    "SKU-400": 11,
}

inventory_report = reconcile_inventory(warehouse_a, warehouse_b)
print(inventory_report)

assert list(inventory_report) == [
    "SKU-100",
    "SKU-200",
    "SKU-300",
    "SKU-400",
]
assert inventory_report["SKU-200"]["status"] == "only_a"
assert inventory_report["SKU-300"]["total"] == 12

{'SKU-100': {'warehouse_a': 12, 'warehouse_b': 8, 'total': 20, 'status': 'both'}, 'SKU-200': {'warehouse_a': 0, 'warehouse_b': 0, 'total': 0, 'status': 'only_a'}, 'SKU-300': {'warehouse_a': 7, 'warehouse_b': 5, 'total': 12, 'status': 'both'}, 'SKU-400': {'warehouse_a': 0, 'warehouse_b': 11, 'total': 11, 'status': 'only_b'}}


This problem separates key presence from quantity. A quantity of zero does not
mean that the SKU is absent.

# Problem 13 — Frequency analysis across many mappings


Find keys occurring in every mapping, any mapping, exactly one mapping, and at
least a chosen number of mappings.

## Solution 13

In [16]:
from collections import Counter


def key_frequency_analysis(*mappings, minimum_count=2):
    if not mappings:
        return {
            "all": set(),
            "any": set(),
            "exactly_one": set(),
            "at_least_minimum": set(),
            "counts": {},
        }

    key_sets = [set(mapping.keys()) for mapping in mappings]

    counts = Counter(
        key
        for key_set in key_sets
        for key in key_set
    )

    return {
        "all": set.intersection(*key_sets),
        "any": set.union(*key_sets),
        "exactly_one": {
            key for key, count in counts.items()
            if count == 1
        },
        "at_least_minimum": {
            key for key, count in counts.items()
            if count >= minimum_count
        },
        "counts": dict(counts),
    }


m1 = {"a": 1, "b": 2, "c": 3}
m2 = {"b": 20, "c": 30, "d": 40}
m3 = {"c": 300, "d": 400, "e": 500}

frequency = key_frequency_analysis(m1, m2, m3, minimum_count=2)

for name in ("all", "any", "exactly_one", "at_least_minimum"):
    print(name, sorted(frequency[name]))

assert frequency["all"] == {"c"}
assert frequency["any"] == {"a", "b", "c", "d", "e"}
assert frequency["exactly_one"] == {"a", "e"}
assert frequency["at_least_minimum"] == {"b", "c", "d"}
assert key_frequency_analysis()["all"] == set()

all ['c']
any ['a', 'b', 'c', 'd', 'e']
exactly_one ['a', 'e']
at_least_minimum ['b', 'c', 'd']


This pattern generalizes pairwise view operations to an arbitrary number of
mappings.

# Problem 14 — Missing keys versus stored None


Compare selected fields in two records. A stored `None` must be treated as a
real value, not as a missing key.

## Solution 14

In [17]:
_MISSING = object()


def compare_selected_fields(before, after, selected_fields):
    missing = {}
    equal = []
    changed = {}

    for field in selected_fields:
        before_value = before.get(field, _MISSING)
        after_value = after.get(field, _MISSING)

        if before_value is _MISSING or after_value is _MISSING:
            missing[field] = {
                "missing_from_before": before_value is _MISSING,
                "missing_from_after": after_value is _MISSING,
            }
        elif before_value == after_value:
            equal.append(field)
        else:
            changed[field] = {
                "before": before_value,
                "after": after_value,
            }

    return {
        "missing": missing,
        "equal": equal,
        "changed": changed,
    }


before_user = {
    "name": "Nora",
    "email": None,
    "preferences": {"theme": "dark"},
    "last_login": "2026-01-01",
}
after_user = {
    "name": "Nora",
    "email": "nora@example.test",
    "preferences": {"theme": "light"},
    "role": "admin",
}

selected = ["name", "email", "preferences", "role", "last_login"]
comparison = compare_selected_fields(before_user, after_user, selected)

print(comparison)

assert comparison["equal"] == ["name"]
assert comparison["changed"]["email"]["before"] is None
assert comparison["missing"]["role"]["missing_from_before"] is True
assert comparison["missing"]["last_login"]["missing_from_after"] is True

{'missing': {'role': {'missing_from_before': True, 'missing_from_after': False}, 'last_login': {'missing_from_before': False, 'missing_from_after': True}}, 'equal': ['name'], 'changed': {'email': {'before': None, 'after': 'nora@example.test'}, 'preferences': {'before': {'theme': 'dark'}, 'after': {'theme': 'light'}}}}


A unique sentinel distinguishes “missing” from every possible stored value,
including `None`.

# Problem 15 — Capstone deployment audit


Create a production-style configuration audit with added, removed, changed, and
unchanged keys. Include numeric deltas, treat booleans separately from numbers,
and redact sensitive values.

## Solution 15

In [18]:
from numbers import Number


SENSITIVE_FRAGMENTS = ("password", "secret", "token", "key")


def is_sensitive_key(key):
    normalized = str(key).lower()
    return any(fragment in normalized for fragment in SENSITIVE_FRAGMENTS)


def redact(key, value):
    if is_sensitive_key(key):
        return "<redacted>"
    return value


def numeric_delta(before_value, after_value):
    if isinstance(before_value, bool) or isinstance(after_value, bool):
        return None
    if isinstance(before_value, Number) and isinstance(after_value, Number):
        return after_value - before_value
    return None


def audit_deployment_config(before, after):
    common = before.keys() & after.keys()

    added = {
        key: redact(key, after[key])
        for key in after
        if key not in before
    }
    removed = {
        key: redact(key, before[key])
        for key in before
        if key not in after
    }

    changed = {}
    unchanged = []

    for key in before:
        if key not in common:
            continue

        before_value = before[key]
        after_value = after[key]

        if before_value == after_value:
            unchanged.append(key)
            continue

        details = {
            "before": redact(key, before_value),
            "after": redact(key, after_value),
            "sensitive": is_sensitive_key(key),
        }

        delta = numeric_delta(before_value, after_value)
        if delta is not None:
            details["delta"] = delta

        changed[key] = details

    return {
        "added": added,
        "removed": removed,
        "changed": changed,
        "unchanged": unchanged,
    }


production_v1 = {
    "replicas": 3,
    "timeout_seconds": 30,
    "debug": False,
    "api_token": "old-token-value",
    "region": "eu-central",
    "legacy_mode": True,
}
production_v2 = {
    "replicas": 5,
    "timeout_seconds": 20,
    "debug": True,
    "api_token": "new-token-value",
    "region": "eu-central",
    "cache_size": 256,
}

deployment_audit = audit_deployment_config(
    production_v1,
    production_v2,
)

print(deployment_audit)

assert deployment_audit["added"] == {"cache_size": 256}
assert deployment_audit["removed"] == {"legacy_mode": True}
assert deployment_audit["changed"]["replicas"]["delta"] == 2
assert deployment_audit["changed"]["timeout_seconds"]["delta"] == -10
assert "delta" not in deployment_audit["changed"]["debug"]
assert deployment_audit["changed"]["api_token"]["before"] == "<redacted>"
assert deployment_audit["changed"]["api_token"]["after"] == "<redacted>"
assert deployment_audit["unchanged"] == ["region"]

{'added': {'cache_size': 256}, 'removed': {'legacy_mode': True}, 'changed': {'replicas': {'before': 3, 'after': 5, 'sensitive': False, 'delta': 2}, 'timeout_seconds': {'before': 30, 'after': 20, 'sensitive': False, 'delta': -10}, 'debug': {'before': False, 'after': True, 'sensitive': False}, 'api_token': {'before': '<redacted>', 'after': '<redacted>', 'sensitive': True}}, 'unchanged': ['region']}


This capstone combines key views with order preservation, type-aware comparison,
security-conscious reporting, and assertions.

# Additional advanced challenge prompts

1. Return shared keys in right-dictionary order.
2. Sort numeric changes by absolute delta.
3. Merge three mappings while recording the source chosen for each key.
4. Compare keys case-insensitively while preserving original spelling.
5. Detect fields whose value type changed.
6. Generate a flat patch and an undo patch.
7. Validate that two mappings have identical keys but intentionally different values.
8. Group shared keys by value hashability.
9. Normalize whitespace in keys before reconciliation.
10. Compare nested mappings recursively and emit dotted paths.

# Compact reference

```python
d.keys()              # live key view
d.values()            # live value view
d.items()             # live item view

d1.keys() | d2.keys() # union -> set
d1.keys() & d2.keys() # intersection -> set
d1.keys() - d2.keys() # difference -> set
d1.keys() ^ d2.keys() # symmetric difference -> set

d1.keys() <= d2.keys()        # subset
d1.keys() >= d2.keys()        # superset
d1.keys().isdisjoint(d2)      # no shared keys

list(d.keys())        # ordered snapshot
set(d.keys())         # unordered unique snapshot
tuple(d.items())      # ordered immutable snapshot
```

In [19]:
def run_final_regression_checks():
    assert common_key_report({"a": 1}, {"a": 2}) == [("a", 1, 2)]
    assert exclusive_items({"a": 0}, {"b": False}) == {"a": 0, "b": False}
    assert classify_changes({}, {}) == {
        "only_left": [],
        "only_right": [],
        "same_value": [],
        "changed_value": [],
    }
    assert diff_mappings({"x": None}, {"x": None})["unchanged"] == ["x"]
    assert merge_with_policy({"x": 1}, {"x": 1}, "error") == {"x": 1}
    assert same_value_multiset({"a": 1, "b": 1}, {"x": 1, "y": 1})
    assert validate_schema({"id": 1}, {"id"})["valid"] is True
    assert compare_selected_fields(
        {"x": None}, {"x": None}, ["x"]
    )["equal"] == ["x"]
    return "All final regression checks passed."


run_final_regression_checks()

'All final regression checks passed.'